In [37]:
%run Table

StatementMeta(, 975695b7-1759-45c2-abe0-74ce5670401d, 119, Finished, Available, Finished, True)

In [12]:
%run ClassManagement

StatementMeta(, 975695b7-1759-45c2-abe0-74ce5670401d, 29, Finished, Available, Finished, True)

In [22]:
import pyspark.sql.functions as F

StatementMeta(, 975695b7-1759-45c2-abe0-74ce5670401d, 71, Finished, Available, Finished, False)

In [34]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

StatementMeta(, 975695b7-1759-45c2-abe0-74ce5670401d, 99, Finished, Available, Finished, False)

In [28]:
class DimDateTransformer:

    def __init__(self):
        self.sources = ClassManagement.get_all_instances(ClassManagement.get_all_subclasses(TableWithDate))
        self.dimDate = DimDate()

    def get_min_date(self):
        min_dates = [source.get_agg_by_column(F.min, source.DATE_COLUMN).date() for source in self.sources]
        min_dates = [d for d in min_dates if d is not None]
        if not min_dates:
            raise Exception("No dates found")
        return min(min_dates)
    
    def get_date_range(self, start_date):
        return spark.range(1).select(
            F.explode(
                F.sequence(F.lit(start_date).cast("date"), F.current_date(), F.expr("INTERVAL 1 DAY"))
            ).alias("date")
        )
    
    def enrich_dates(self, dates):
        return dates.select(
            F.col("date"),
            F.year("date").cast("smallint").alias("year"),
            F.month("date").cast("smallint").alias("month"),
            F.date_format("date", "MMMM").alias("month_name"),
            F.dayofmonth("date").cast("smallint").alias("day"),
            (F.weekday("date")+1).cast("smallint").alias("day_of_week"),
            F.date_format("date", "EEEE").alias("day_name"),
            F.when(F.month("date").isin(12, 1, 2), "Winter")
                .when(F.month("date").isin(3, 4, 5), "Spring")
                .when(F.month("date").isin(6, 7, 8), "Summer")
                .otherwise("Autumn").alias("season")
        )
    
    def calculate_dim_date(self):
        start_date = self.get_min_date()
        if start_date > date.today():
            raise Exception("Start day greater than today")
        dates = self.get_date_range(start_date)
        return self.enrich_dates(dates)
    
    def transform(self):
        new_df = self.calculate_dim_date()
        self.dimDate.insert_new(new_df)


StatementMeta(, 975695b7-1759-45c2-abe0-74ce5670401d, 93, Finished, Available, Finished, False)

In [38]:
DimDateTransformer().transform()

StatementMeta(, 975695b7-1759-45c2-abe0-74ce5670401d, 120, Finished, Available, Finished, False)

TypeError: '>' not supported between instances of 'datetime.datetime' and 'datetime.date'